In [ ]:
import importlib
import import_packages # all my packages are there, it's easier
importlib.reload(import_packages) # so I can add new packages in import_packages.py without restarting the kernel
from import_packages import *
print("Packages (re)loaded successfully!")

# Supplementary Figures

Standalone illustration figures for the paper/presentations - not part of the core
statistical analysis pipeline. Generates example stimuli and schematic diagrams (Gabor
patch, band-limited noise mask, CSF/AUC illustration, dot-display mockup), plus brain
mask/atlas overlay figures (V1 and M1 ROIs on the MNI template with the Julich-Brain
atlas outline). The brain-mask cells need external MRI mask/atlas files not included
in this repo (see the `<path-to-external-data>` placeholders).

### Gabor patch stimulus example

In [ ]:
def generate_gabor(size=128, frequency=0.1, theta=0, sigma=20, phase=0, contrast=1.0):
    """Generate a Gabor patch."""
    x = np.linspace(-size / 2, size / 2, size)
    y = np.linspace(-size / 2, size / 2, size)
    X, Y = np.meshgrid(x, y)

    # Rotate coordinates
    theta_rad = np.deg2rad(theta)
    X_theta = X * np.cos(theta_rad) + Y * np.sin(theta_rad)

    # Gabor formula
    sinusoid = np.cos(2 * np.pi * frequency * X_theta + np.deg2rad(phase))
    gaussian = np.exp(-(X**2 + Y**2) / (2 * sigma**2))
    gabor = sinusoid * gaussian

    # Normalize to [0, 1] and map to [0, 1] with mid-grey (0.5) as background
    gabor_normalized = (contrast*gabor / np.max(np.abs(gabor)) / 2 + 0.5)
    return gabor_normalized



In [ ]:
# Plot the Gabor patch
gabor = generate_gabor(size=200, frequency=0.04, phase=45, theta=45, sigma=30, contrast=0.3)
plt.imshow(gabor, cmap='gray', vmin=0, vmax=1)
plt.axis('off')
#plt.title("Gabor Patch")
plt.show()

# Plot the Gabor patch
gabor = generate_gabor(size=200, frequency=0.04, theta=45, sigma=30, contrast=1)
plt.imshow(gabor, cmap='gray', vmin=0, vmax=1)
plt.axis('off')
#plt.title("Gabor Patch")
plt.show()

### Band-limited noise stimulus example

In [ ]:
import numpy as np

def generate_band_limited_noise(
    size=128,
    center_frequency=0.1,   # cycles per pixel
    bandwidth=0.02,         # frequency-domain std (cycles per pixel)
    contrast=1.0,
    seed=None
):
    """
    Generate full-field isotropic band-limited noise.

    Returns an image in [0, 1] with mid-grey (0.5) background,
    suitable for use as a noise masker.
    """
    if seed is not None:
        np.random.seed(seed)

    # --- White noise ---
    noise = np.random.randn(size, size)

    # --- Fourier transform ---
    F = np.fft.fftshift(np.fft.fft2(noise))

    # --- Frequency coordinates (cycles per pixel) ---
    fx = np.fft.fftshift(np.fft.fftfreq(size))
    fy = np.fft.fftshift(np.fft.fftfreq(size))
    FX, FY = np.meshgrid(fx, fy)
    R = np.sqrt(FX**2 + FY**2)

    # --- Radial band-pass filter ---
    bandpass = np.exp(-(R - center_frequency)**2 / (2 * bandwidth**2))

    # --- Apply filter ---
    F_filtered = F * bandpass
    noise_filtered = np.fft.ifft2(np.fft.ifftshift(F_filtered)).real

    # --- Normalize (zero mean, unit max abs) ---
    noise_filtered -= np.mean(noise_filtered)
    noise_filtered /= np.max(np.abs(noise_filtered))

    # --- Apply contrast and map to [0, 1] ---
    noise_image = contrast * noise_filtered / 2.0 + 0.5

    return noise_image


In [ ]:
noise = generate_band_limited_noise(
    size=150,
    center_frequency=0.03,
    bandwidth=0.02,
    contrast=1.0,
    seed=42
)

plt.imshow(noise, cmap='gray', vmin=0, vmax=1)
plt.axis('off')
#plt.title("Gabor Patch")
plt.savefig('<path-to-external-data>/Data/Plots/new_new/noise.png', dpi=600, bbox_inches='tight')

plt.show()


### CSF / Area-Between-Curves illustration

In [ ]:
from scipy.optimize import curve_fit

# Define Gaussian function
def gaussian(x, A, mu, sigma):
    return A * np.exp(-(x - mu)**2 / (2 * sigma**2))

# Original data points
x_vals = np.array([50/16, 50/8, 50/4, 50/2, 50*(2/3), 50])
y_medium = np.array([20, 40, 20, 10, 1, 1])
y_high   = np.array([25, 80, 60, 20, 10, 1])

# Fit Gaussian curves
params_medium, _ = curve_fit(gaussian, x_vals, y_medium, p0=[50, 25, 10])
params_high, _ = curve_fit(gaussian, x_vals, y_high, p0=[80, 25, 10])

# Smooth x values
x_smooth = np.linspace(x_vals.min(), x_vals.max(), 300)
y_medium_fit = gaussian(x_smooth, *params_medium)
y_high_fit = gaussian(x_smooth, *params_high)

# Create vertically stacked subplots
fig, axes = plt.subplots(2, 1, figsize=(5, 6), sharex=True, sharey=True)

# Medium AUC plot
axes[0].plot(x_smooth, y_medium_fit, color='#D55E00', lw=2)
axes[0].fill_between(x_smooth, y_medium_fit, 1, color='#D55E00', alpha=0.4)
axes[0].set_yscale('log')
axes[0].set_ylabel("Sensitivity")
axes[0].text(0.98, 0.92, "Medium AUC", transform=axes[0].transAxes,
             fontsize=18, ha='right', va='top', color='#D55E00')

# High AUC plot
axes[1].plot(x_smooth, y_high_fit, color='#009E73', lw=2)
axes[1].fill_between(x_smooth, y_high_fit, 1, color='#009E73', alpha=0.4)
axes[1].set_yscale('log')
axes[1].set_ylabel("Sensitivity")
axes[1].text(0.98, 0.92, "Large AUC", transform=axes[1].transAxes,
             fontsize=18, ha='right', va='top', color='#009E73')

# Formatting for both axes
for ax in axes:
    ax.set_xlim(50/16, 36)
    ax.set_ylim(1, 100)
    ax.set_yticks([1, 10, 100])
    ax.grid(False)
    ax.tick_params(axis='x', labelbottom=False)  # Remove x tick labels

# Remove overall x-axis label
fig.supxlabel("")  # Ensures no x-axis label is shown

plt.tight_layout()
plt.show()

In [ ]:
from matplotlib.patches import Patch

# Reuses gaussian, x_smooth, y_medium_fit, y_high_fit from the previous cell

# Create combined plot
fig, ax = plt.subplots(figsize=(4, 3))

# Plot green (background) first
ax.plot(x_smooth, y_high_fit, color="#000000", lw=2)
ax.fill_between(x_smooth, y_high_fit, 1, color="#9B48DA", alpha=0.6)

# Then plot orange (foreground)
# ax.plot(x_smooth, y_medium_fit, color="#000000", lw=2)
# ax.fill_between(x_smooth, y_medium_fit, 1, color="#1500D5", alpha=0.4)


# Y-axis formatting
ax.set_yscale('log')
ax.set_xlim(50/16, 36)
ax.set_xticks([4, 18, 35])
ax.set_xticklabels(['1', '25', '50'])
ax.set_ylim(1, 100)
ax.set_yticks([1, 10, 100])
ax.set_yticklabels(['1', '10', '100'])

# Misc formatting
ax.grid(False)
ax.set_ylabel("Sensitivity")
ax.set_xlabel("Spatial Frequency")

# ax.tick_params(axis='x', labelbottom=False)

# # Custom legend with color boxes
# legend_elements = [
#     Patch(facecolor='#D55E00', label='Pre-training AUC'),
#     Patch(facecolor='#009E73', label='Post-training AUC')
# ]
# ax.legend(handles=legend_elements, loc='upper right', fontsize=11)

plt.tight_layout()
plt.savefig('<path-to-external-data>/Data/Plots/Narrative_plots/v3/AUC_example.png', dpi=600, bbox_inches='tight')

plt.show()

In [ ]:
# Create minimalist plot
fig, ax = plt.subplots(figsize=(3.4, 3))

# Green curve (background)
ax.plot(x_smooth, y_high_fit, color='#009E73', lw=2)
ax.fill_between(x_smooth, y_high_fit, 1, color='#009E73', alpha=1)

# White curve (foreground)
ax.plot(x_smooth, y_medium_fit, color='white', lw=2)
ax.fill_between(x_smooth, y_medium_fit, 1, color='white', alpha=1)

# Remove all markings
ax.set_yscale('log')
ax.set_xlim(50/16, 36)
ax.set_ylim(1, 100)
ax.set_xticks([])
ax.set_yticks([])
ax.set_yticklabels([])
ax.set_xticklabels([])
ax.set_ylabel("")
ax.set_xlabel("")
ax.tick_params(left=False, bottom=False)
ax.grid(False)
ax.legend().set_visible(False)
# Remove the grey frame (spines)
for spine in ax.spines.values():
    spine.set_visible(False)

# Add custom text box inside the white curve
ax.text(
    7, 4,  # (x, y) position — adjust based on your data and where the white curve sits
    "Pre AUC - Post AUC \n                = \n    AUC difference",

    fontsize=11,
    color='black',
    bbox=dict(facecolor='white', edgecolor='black', boxstyle='round,pad=0.3', alpha=0.9)
)
plt.tight_layout()
plt.show()

### Dot-display mockup

In [ ]:
import matplotlib.patches as patches

# Figure setup
dpi = 100
width, height = 1600, 1200
fig = plt.figure(figsize=(width / dpi, height / dpi), dpi=dpi)
ax = plt.axes([0, 0, 1, 1])
plt.axis("off")

# Draw black background
background = patches.Rectangle((0, 0), width, height, color='black')
ax.add_patch(background)

# Settings
dot_radius = 10  # 20px diameter
quadrant_width, quadrant_height = 700, 500
margin = 50
dots_per_quadrant = 150  # adjustable

# Define quadrant origins
quadrants = {
    "bottom_left": (margin, margin),
    "top_right": (width - margin - quadrant_width, margin),
    "top_left": (margin, height - margin - quadrant_height),
    "bottom_right": (width - margin - quadrant_width, height - margin - quadrant_height)
}

# Helper to add a dot
def add_dot(x, y, color, alpha=1.0):
    circle = plt.Circle((x, y), radius=dot_radius, color=color, alpha=alpha)
    ax.add_patch(circle)

# Plotting
for name, (x0, y0) in quadrants.items():
    if name != "top_right":
        # Standard quadrants
        for _ in range(dots_per_quadrant):
            x = np.random.randint(x0, x0 + quadrant_width)
            y = np.random.randint(y0, y0 + quadrant_height)
            color = np.random.rand(3,)
            add_dot(x, y, color)
    else:
        # Top-right quadrant: draw only inside central 450x300 area
        central_width, central_height = 450, 300
        
        for _ in range(150):
            x = np.random.randint(x0, x0 + quadrant_width)
            y = np.random.randint(y0, y0 + quadrant_height)
            color = np.random.rand(3,)
            add_dot(x, y, color)
            cx0 = x0 + (quadrant_width - central_width) // 2
            cy0 = y0 + (quadrant_height - central_height) // 2
            rectangle = patches.Rectangle((cx0, cy0), central_width, central_height, facecolor='black', edgecolor='red', linewidth=3)
            ax.add_patch(rectangle)
        for _ in range(50):
            cx0 = x0 + (quadrant_width - central_width) // 2
            cy0 = y0 + (quadrant_height - central_height) // 2
            x = np.random.randint(cx0, cx0 + central_width)
            y = np.random.randint(cy0, cy0 + central_height)
            color = np.random.rand(3,)
            add_dot(x, y, color)  # Original dot
            add_dot(x + 20, y, color, alpha=0.5)  # Shifted duplicate

# Final display setup
plt.xlim(0, width)
plt.ylim(0, height)
plt.gca().invert_yaxis()

plt.show()


### Brain mask / atlas overlay figures
Requires external MRI mask and Julich-Brain atlas files (not included in this repo).

In [ ]:
from nilearn import plotting, image, datasets
from nilearn.plotting import plot_roi, show
import nibabel as nib

mask = nib.load("<path-to-external-data>/extra_MRI/masks/V1_voxel_sum_S1.nii.gz")
mni = datasets.load_mni152_template()
mask_res = image.resample_to_img(mask, mni, interpolation='nearest')

# glass brain (projection)

# display = plotting.plot_glass_brain(mask_res, colorbar=True, plot_abs=True, display_mode="lyrz")
# display.savefig('<path-to-external-data>/extra_MRI/output/V1_S1_mask.png', dpi=600)
# Remember to close the display
# display.close()

# display = plot_roi(mask_res, colorbar=True, draw_cross=False, display_mode='xz', cmap="bwr", vmin=0, vmax=18, cbar_tick_format="%d")
# display.savefig('<path-to-external-data>/extra_MRI/output/V1_S1_mask.png', dpi=600)
plot_roi(mask_res, colorbar=True, draw_cross=False, display_mode='xz', cmap="bwr", vmin=0, vmax=18, cbar_tick_format="%d")
# Remember to close the display
# display.close()

In [ ]:
import xml.etree.ElementTree as ET

# --- 1) Parse XML into a dictionary ---
julich_xml = "<path-to-external-data>/extra_MRI/julich/maximum-probability-maps_MPMs_207-areas/JulichBrainAtlas_3.1_207areas_MPM_lh_MNI152.xml"


tree = ET.parse(julich_xml)
root = tree.getroot()
structures = root.find('Structures')

# build name -> (num, id, grayvalue) mapping
mapping = {}
for s in structures.findall('Structure'):
    name = s.text.strip()
    num = int(s.attrib.get('num'))
    idv = int(s.attrib.get('id'))
    gray = int(s.attrib.get('grayvalue'))
    mapping[name] = {'num': num, 'id': idv, 'grayvalue': gray}

# Print the entries we're interested in
targets = ["Area 6d1 (PreCG)", "Area 6d2 (PreCG)", "Area 6d3 (SFS)"]
for t in targets:
    print(t, "->", mapping.get(t))

In [ ]:
# Jupyter-friendly plotting + saving with white background + both V1 labels outlined
from nilearn import plotting, image, datasets
from nilearn.plotting import plot_roi
from nilearn.image import math_img
import nibabel as nib
from matplotlib.colors import LinearSegmentedColormap
import matplotlib.pyplot as plt
import numpy as np

# --- your files (update paths) ---
mask_path = "<path-to-external-data>/extra_MRI/masks/V1_voxel_sum_S1.nii.gz"
out_png = "<path-to-external-data>/extra_MRI/output/V1_S1_mask_with_glasser_outline.png"

lh_path = "<path-to-external-data>/extra_MRI/julich/maximum-probability-maps_MPMs_207-areas/JulichBrainAtlas_3.1_207areas_MPM_lh_MNI152.nii.gz"
rh_path = "<path-to-external-data>/extra_MRI/julich/maximum-probability-maps_MPMs_207-areas/JulichBrainAtlas_3.1_207areas_MPM_rh_MNI152.nii.gz"

lh = nib.load(lh_path)
rh = nib.load(rh_path)

lh_res = image.resample_img(
    lh,
    target_affine=mni.affine,
    target_shape=mni.shape,
    interpolation='nearest'   # preserve integer labels
)
rh_res = image.resample_img(
    rh,
    target_affine=mni.affine,
    target_shape=mni.shape,
    interpolation='nearest'   # preserve integer labels
)

# v1_bilateral = math_img("(img1 == 91) | (img2 == 91)", img1=lh_res, img2=rh_res)

# --- load and resample to MNI ---
mask = nib.load(mask_path)
mni = datasets.load_mni152_template()      # background image
mask_res = image.resample_to_img(mask, mni, interpolation='nearest')

# --- custom orange -> yellow colormap ---
orange = "#ff7f00"
middle = "#ffb347"
yellow = "#ffd700"
cmap_orange_yellow = LinearSegmentedColormap.from_list("orange_to_yellow", [orange, middle, yellow], N=256)

# --- plot: ROI (filled) + contours (outline of Glasser V1) ---
display = plot_roi(
    mask_res,
    bg_img=mni,                # show MNI background
    cmap=cmap_orange_yellow,
    display_mode='x',
    cut_coords=[2],
    draw_cross=False,
    colorbar=True,
    vmin=0, vmax=18,
    cbar_tick_format="%d", 
    black_bg=False
)

# add black outline for the combined V1 (both hemispheres)
# display.add_contours(v1_bilateral, colors='black', linewidths=1.0, levels=[0.5])

# --- make sure figure background is white (for saving) ---
# get the current matplotlib figure used by nilearn and set facecolor
fig = plt.gcf()
fig.patch.set_facecolor('white')

# --- save with white background, high DPI ---
display.savefig(out_png, dpi=600, transparent=False)

# --- show inline in Jupyter ---
# plotting.show() calls plt.show() internally; that will render the figure in the notebook
plotting.show()

# NOTE: don't call display.close() before plotting.show() — we want it visible.
# when you're done and want to clear the display, you can do:
# display.close()

In [ ]:
# Jupyter-friendly plotting + saving with white background + both V1 labels outlined
from nilearn import plotting, image, datasets
from nilearn.plotting import plot_roi
from nilearn.image import math_img
import nibabel as nib
from matplotlib.colors import LinearSegmentedColormap
import matplotlib.pyplot as plt
import numpy as np

# --- your files (update paths) ---
mask_path = "<path-to-external-data>/extra_MRI/masks/M1_voxel_sum_S1.nii.gz"
out_png = "<path-to-external-data>/extra_MRI/output/M1_S1_mask_with_glasser_outline.png"

lh_path = "<path-to-external-data>/extra_MRI/julich/maximum-probability-maps_MPMs_207-areas/JulichBrainAtlas_3.1_207areas_MPM_lh_MNI152.nii.gz"
rh_path = "<path-to-external-data>/extra_MRI/julich/maximum-probability-maps_MPMs_207-areas/JulichBrainAtlas_3.1_207areas_MPM_rh_MNI152.nii.gz"

lh = nib.load(lh_path)
rh = nib.load(rh_path)

lh_res = image.resample_img(
    lh,
    target_affine=mni.affine,
    target_shape=mni.shape,
    interpolation='nearest'   # preserve integer labels
)
rh_res = image.resample_img(
    rh,
    target_affine=mni.affine,
    target_shape=mni.shape,
    interpolation='nearest'   # preserve integer labels
)

# v1_bilateral = math_img("(img1 == 1) | (img2 == 30) | (img2 == 57) | (img2 == 104)", img1=lh_res, img2=lh_res)

# --- load and resample to MNI ---
mask = nib.load(mask_path)
mni = datasets.load_mni152_template()      # background image
mask_res = image.resample_to_img(mask, mni, interpolation='nearest')

# --- custom orange -> yellow colormap ---
orange = "#0051ff"
middle = "#417bf6"
yellow = "#a1bcf7"
cmap_orange_yellow = LinearSegmentedColormap.from_list("orange_to_yellow", [orange, middle, yellow], N=256)

# --- plot: ROI (filled) + contours (outline of Glasser V1) ---
display = plot_roi(
    mask_res,
    bg_img=mni,                # show MNI background
    cmap=cmap_orange_yellow,
    display_mode='x',
    cut_coords=[-30],
    draw_cross=False,
    colorbar=True,
    vmin=0, vmax=18,
    cbar_tick_format="%d", 
    black_bg=False
)

# add black outline for the combined V1 (both hemispheres)
# display.add_contours(v1_bilateral, colors='black', linewidths=1.0, levels=[0.5])

# --- make sure figure background is white (for saving) ---
# get the current matplotlib figure used by nilearn and set facecolor
fig = plt.gcf()
fig.patch.set_facecolor('white')

# --- save with white background, high DPI ---
display.savefig(out_png, dpi=600, transparent=False)

# --- show inline in Jupyter ---
# plotting.show() calls plt.show() internally; that will render the figure in the notebook
plotting.show()

# NOTE: don't call display.close() before plotting.show() — we want it visible.
# when you're done and want to clear the display, you can do:
# display.close()

In [ ]:
from nilearn import plotting, image, datasets
from nilearn.plotting import plot_roi, show
import nibabel as nib

mask = nib.load("<path-to-external-data>/extra_MRI/masks/sub_10_example/S1_V1_mask_standard.nii.gz")
mni = datasets.load_mni152_template()
mask_res = image.resample_to_img(mask, mni, interpolation='nearest')

# glass brain (projection)

# display = plotting.plot_glass_brain(mask_res, colorbar=True, plot_abs=True, display_mode="lyrz")
# display.savefig('<path-to-external-data>/extra_MRI/output/V1_S1_mask.png', dpi=600)
# Remember to close the display
# display.close()

display = plot_roi(mask_res, colorbar=True, draw_cross=False, display_mode='z', cmap="bwr", cut_coords=[11, 16], vmin=0, vmax=1)
display.savefig('<path-to-external-data>/extra_MRI/output/V1_example_mask.png', dpi=600)
# Remember to close the display
display.close()

In [ ]:
from nilearn import plotting, image, datasets
from nilearn.plotting import plot_roi, show
import nibabel as nib

mask = nib.load("<path-to-external-data>/extra_MRI/masks/sub_10_example/S1_M1_mask_standard.nii.gz")
mni = datasets.load_mni152_template()
mask_res = image.resample_to_img(mask, mni, interpolation='nearest')

# glass brain (projection)

# display = plotting.plot_glass_brain(mask_res, colorbar=True, plot_abs=True, display_mode="lyrz")
# display.savefig('<path-to-external-data>/extra_MRI/output/V1_S1_mask.png', dpi=600)
# Remember to close the display
# display.close()

display = plot_roi(mask_res, colorbar=True, draw_cross=False, display_mode='z', cmap="bwr", cut_coords=[40, 49], vmin=0, vmax=1)
display.savefig('<path-to-external-data>/extra_MRI/output/M1_example_mask.png', dpi=600)
# Remember to close the display
display.close()